In [ ]:
"""
Series-Level Time Series Anomaly Detection Pipeline
- Predicts whether an entire series is anomalous (y_i=1) or normal (y_i=0)
- Aggregates point-level data into series-level statistical features
- Uses metadata labels if available, otherwise infers from max(point_label)
- Optimized for standard tabular ML workflows with HistGradientBoosting
"""
import gc
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, roc_auc_score, precision_recall_curve, f1_score, auc
import joblib
import logging
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Optional
import warnings
import argparse

warnings.filterwarnings('ignore')

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-8s | %(message)s',
    handlers=[logging.StreamHandler()]
)
logger = logging.getLogger(__name__)

DTYPE_FLOAT = np.float32

@dataclass
class Config:
    data_path: Path
    metadata_path: Optional[Path] = None
    model_dir: Path = Path("models_series-level")
    output_path: Optional[Path] = None
    test_size: float = 0.15
    val_size: float = 0.15
    random_state: int = 42
    train_sample_frac: float = 1.0

class SeriesFeatureExtractor:
    """Converts point-level time series data into series-level statistical features."""
    
    @staticmethod
    def aggregate(df: pd.DataFrame) -> pd.DataFrame:
        logger.info("Aggregating point-level data to series-level features...")
        
        # Sort to ensure correct diff calculations per series
        df = df.sort_values(['series_id', 'time_index'])
        
        # Precompute point-level transformations
        df['abs_diff'] = df.groupby('series_id')['value'].diff().abs()
        
        # Aggregate basic statistics
        value_stats = df.groupby('series_id')['value'].agg([
            'mean', 'std', 'min', 'max', 'median', 'skew', 'kurt', 'count'
        ]).rename(columns={c: f'value_{c}' for c in ['mean', 'std', 'min', 'max', 'median', 'skew', 'kurt', 'count']})
        
        # Aggregate difference statistics
        diff_stats = df.groupby('series_id')['abs_diff'].agg([
            'mean', 'max', 'std'
        ]).fillna(0).rename(columns={c: f'diff_{c}' for c in ['mean', 'max', 'std']})
        
        # Combine and derive higher-level features
        features = pd.concat([value_stats, diff_stats], axis=1)
        features['range'] = features['value_max'] - features['value_min']
        features['cv'] = features['value_std'] / (features['value_mean'].abs() + 1e-8)
        features['diff_ratio'] = features['diff_mean'] / (features['value_mean'].abs() + 1e-8)
        
        # Clean numerical artifacts
        features = features.replace([np.inf, -np.inf], np.nan)
        
        # Assign series label: use max of point labels (if any point is anomalous, series is anomalous)
        if 'label' in df.columns:
            features['label'] = df.groupby('series_id')['label'].max().astype(np.int8)
        else:
            features['label'] = 0
            
        features = features.reset_index()
        features = features.astype({c: DTYPE_FLOAT for c in features.select_dtypes(include='float64').columns})
        
        logger.info(f"Aggregated to {len(features)} series with {len(features.columns)-2} features")
        return features

class SeriesAnomalyPipeline:
    """Pipeline for series-level anomaly classification."""
    
    def __init__(self, config: Config):
        self.config = config
        self.feature_cols: List[str] = []
        self.pipeline: Optional[Pipeline] = None
        self.threshold: float = 0.5
        self._build_pipeline()
        
    def _build_pipeline(self):
        self.pipeline = Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', RobustScaler()),
            ('model', HistGradientBoostingClassifier(
                max_depth=6,
                learning_rate=0.05,
                max_iter=500,
                class_weight='balanced',
                random_state=self.config.random_state,
                validation_fraction=0.1,
                early_stopping=True,
                n_iter_no_change=20,
                max_bins=64,
                min_samples_leaf=50,
                l2_regularization=0.1
            ))
        ])
        
    def _split_data(self, features: pd.DataFrame) -> tuple:
        logger.info("Splitting series data...")
        X = features[self.feature_cols].values.astype(DTYPE_FLOAT)
        y = features['label'].values.astype(np.int8)
        
        X_trainval, X_test, y_trainval, y_test = train_test_split(
            X, y, test_size=self.config.test_size, random_state=self.config.random_state, stratify=y
        )
        
        val_ratio = self.config.val_size / (1 - self.config.test_size)
        X_train, X_val, y_train, y_val = train_test_split(
            X_trainval, y_trainval, test_size=val_ratio, random_state=self.config.random_state, stratify=y_trainval
        )
        
        logger.info(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")
        logger.info(f"Train anomaly ratio: {y_train.mean():.4f}")
        return X_train, y_train, X_val, y_val, X_test, y_test
        
    def _optimize_threshold(self, y_true: np.ndarray, y_prob: np.ndarray) -> float:
        precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
        f1_scores = 2 * (precision * recall) / (precision + recall + 1e-10)
        best_idx = np.argmax(f1_scores)
        best_thr = thresholds[min(best_idx, len(thresholds) - 1)]
        return float(np.clip(best_thr, 0.01, 0.99))
        
    def train(self, df: pd.DataFrame, metadata_df: Optional[pd.DataFrame] = None) -> Dict:
        # 1. Feature extraction
        features = SeriesFeatureExtractor.aggregate(df)
        
        # 2. Merge metadata if provided (y_i is the ground-truth series label)
        if metadata_df is not None and 'y_i' in metadata_df.columns:
            meta_labels = metadata_df[['series_id', 'y_i']].drop_duplicates('series_id')
            features = features.merge(meta_labels, on='series_id', how='left')
            features['label'] = features['y_i'].fillna(features['label']).astype(np.int8)
            features.drop(columns=['y_i'], inplace=True, errors='ignore')
            
        # 3. Sampling (if needed)
        if self.config.train_sample_frac < 1.0:
            logger.info(f"Sampling {self.config.train_sample_frac*100:.0f}% of series...")
            features = features.groupby('label', group_keys=False).sample(
                frac=self.config.train_sample_frac, random_state=self.config.random_state
            ).reset_index(drop=True)
            
        self.feature_cols = [c for c in features.columns if c not in ['series_id', 'label']]
        X_train, y_train, X_val, y_val, X_test, y_test = self._split_data(features)
        
        # 4. Train
        self.pipeline.fit(X_train, y_train)
        
        # 5. Validation & threshold optimization
        val_prob = self.pipeline.predict_proba(X_val)[:, 1]
        self.threshold = self._optimize_threshold(y_val, val_prob)
        logger.info(f"Optimal threshold (F1): {self.threshold:.4f}")
        
        val_pred = (val_prob >= self.threshold).astype(int)
        logger.info("Validation metrics:\n" + classification_report(y_val, val_pred, zero_division=0))
        
        # 6. Test evaluation
        test_prob = self.pipeline.predict_proba(X_test)[:, 1]
        test_pred = (test_prob >= self.threshold).astype(int)
        
        prec_curve, rec_curve, _ = precision_recall_curve(y_test, test_prob)
        pr_auc = auc(rec_curve, prec_curve)
        
        metrics = {
            'roc_auc': roc_auc_score(y_test, test_prob),
            'pr_auc': pr_auc,
            'f1': f1_score(y_test, test_pred, zero_division=0),
            'precision': y_test[test_pred == 1].mean() if test_pred.sum() > 0 else 0.0,
            'recall': test_pred[y_test == 1].mean() if y_test.sum() > 0 else 0.0
        }
        logger.info("Test metrics:\n" + classification_report(y_test, test_pred, zero_division=0))
        logger.info(f"Test ROC-AUC: {metrics['roc_auc']:.4f}, PR-AUC: {metrics['pr_auc']:.4f}")
        return metrics
        
    def predict(self, df: pd.DataFrame) -> pd.DataFrame:
        features = SeriesFeatureExtractor.aggregate(df)
        X = features[self.feature_cols].values.astype(DTYPE_FLOAT)
        probs = self.pipeline.predict_proba(X)[:, 1]
        preds = (probs >= self.threshold).astype(int)
        
        out = features[['series_id']].copy()
        out['pred_label'] = preds
        out['pred_prob'] = probs
        return out
        
    def save(self):
        self.config.model_dir.mkdir(parents=True, exist_ok=True)
        artifacts = {
            'pipeline': self.pipeline,
            'feature_cols': self.feature_cols,
            'threshold': self.threshold,
            'config': self.config
        }
        path = self.config.model_dir / "anomaly_model.joblib"
        joblib.dump(artifacts, path)
        logger.info(f"Model saved to: {path}")
        
    def load(self):
        path = self.config.model_dir / "anomaly_model.joblib"
        if not path.exists():
            raise FileNotFoundError(f"Model not found: {path}")
        artifacts = joblib.load(path)
        self.pipeline = artifacts['pipeline']
        self.feature_cols = artifacts['feature_cols']
        self.threshold = artifacts['threshold']
        self.config = artifacts['config']
        logger.info("Model loaded successfully.")

def main():
    parser = argparse.ArgumentParser(description='Series-Level Anomaly Detection')
    parser.add_argument('mode', nargs='?', default='train', choices=['train', 'predict'], help='Execution mode')
    parser.add_argument('--data', type=Path, default=Path('../data/R1.parquet'), help='Path to point-level data')
    parser.add_argument('--metadata', type=Path, default=Path("../data/R1_metadata.parquet"), help='Path to metadata parquet (optional)')
    parser.add_argument('--model-dir', type=Path, default=Path('models_series-level'), help='Model directory')
    parser.add_argument('--output', type=Path, default=Path("result_series-level"), help='Path for predictions')
    
    args, _ = parser.parse_known_args()
    config = Config(
        data_path=args.data, 
        metadata_path=args.metadata, 
        model_dir=args.model_dir, 
        output_path=args.output
    )
    detector = SeriesAnomalyPipeline(config)
    
    if args.mode == 'train':
        logger.info("=== TRAINING ===")
        if not config.data_path.exists():
            raise FileNotFoundError(f"Data file not found: {config.data_path}")
            
        df = pd.read_parquet(config.data_path)
        metadata_df = pd.read_parquet(config.metadata_path) if config.metadata_path and config.metadata_path.exists() else None
        
        metrics = detector.train(df, metadata_df)
        detector.save()
        logger.info(f"Final metrics: {metrics}")
        
    elif args.mode == 'predict':
        logger.info("=== PREDICTION ===")
        detector.load()
        df = pd.read_parquet(config.data_path)
        preds = detector.predict(df)
        out_path = config.output_path if config.output_path else Path("series_predictions.parquet")
        preds.to_parquet(out_path, index=False)
        logger.info(f"Predictions saved to: {out_path}")
        logger.info(f"Label distribution: {preds['pred_label'].value_counts().to_dict()}")  

if __name__ == '__main__':
    main()

2026-05-23 15:43:01,255 | INFO     | === TRAINING ===
2026-05-23 15:43:02,431 | INFO     | Aggregating point-level data to series-level features...
2026-05-23 15:43:10,275 | INFO     | Aggregated to 4768 series with 14 features
2026-05-23 15:43:10,383 | INFO     | Splitting series data...
2026-05-23 15:43:10,407 | INFO     | Train: 3336, Val: 716, Test: 716
2026-05-23 15:43:10,410 | INFO     | Train anomaly ratio: 0.3891
2026-05-23 15:43:10,895 | INFO     | Optimal threshold (F1): 0.5057
2026-05-23 15:43:10,905 | INFO     | Validation metrics:
              precision    recall  f1-score   support

           0       0.95      0.93      0.94       438
           1       0.89      0.92      0.91       278

    accuracy                           0.93       716
   macro avg       0.92      0.93      0.92       716
weighted avg       0.93      0.93      0.93       716

2026-05-23 15:43:10,927 | INFO     | Test metrics:
              precision    recall  f1-score   support

           0     